## Example: NER Fine-Tuning with OpenAI Using SFT + DPO 

This notebook demonstrates how to fine-tune an NER task using **Supervised Fine-Tuning (SFT)** followed by **Directed Preference Optimization (DPO)**, based on the `tensorzero` framework and OpenAI APIs. 

### Setup

In [ ]:

#!pip install tensorzero python-dotenv pandas
import os
from dotenv import load_dotenv

load_dotenv()  # Loads the .env file containing your OPENAI_API_KEY 


### Load and Prepare Dataset for SFT

In [ ]:
import pandas as pd
import json

NUM_TRAIN_DATAPOINTS = 500
NUM_VAL_DATAPOINTS = 500

# Helper function to convert a row into prompt-completion format
def to_prompt_completion(row):
    return {
        "prompt": row["input"],
        "completion": json.dumps(row["output"])
    }

# Function to load and process the NER dataset
def load_ner_dataset(path: str):
    df = pd.read_csv(path)
    df["output"] = df["output"].apply(json.loads)

    train_df = df[df["split"] == 0].sample(frac=1, random_state=0).reset_index(drop=True)
    val_df = df[df["split"] == 1].sample(frac=1, random_state=0).reset_index(drop=True)

    train_df = train_df.iloc[:NUM_TRAIN_DATAPOINTS]
    val_df = val_df.iloc[:NUM_VAL_DATAPOINTS]

    # Convert DataFrame rows to prompt–completion format
    train_sft = train_df.apply(to_prompt_completion, axis=1).tolist()
    val_sft = val_df.apply(to_prompt_completion, axis=1).tolist()

    return train_sft, val_sft

In [ ]:
train_sft, val_sft = load_ner_dataset("examples/data-extraction-ner/data/conllpp.csv")
print(f"Number of training samples: {len(train_sft)}")
print(f"Number of validation samples: {len(val_sft)}")

# Optional: preview a sample
print("Example training sample:", train_sft[0])